In [1]:
from fastapi import FastAPI #creates a webserver API
from pydantic import BaseModel
import os
import json
from dotenv import load_dotenv
from openai import OpenAI


In [2]:
load_dotenv(override = True)
google_api_key = os.getenv("GOOGLE_API_KEY")
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

Google API Key exists and begins AIzaSyCx


In [3]:
client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("GOOGLE_API_KEY")
)

MODEL = "gemini-2.5-flash"

system_message = """
You are a helpful assistant for an Airline called Airticket.
Give short, courteous answers, no more than 2 sentences.
Always be accurate. If you don't know the answer, say so.
"""



In [4]:
def get_ticket_price(destination_city):
    prices = {
        "London": "$850",
        "Dubai": "$600",
        "New York": "$1200"
    }
    return prices.get(destination_city, "Price not available")


price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the customer wants to travel to"
            }
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": price_function}
]

In [6]:
#Creating Fast API, initializes backend server
app = FastAPI()

chat_history = []

class ChatRequest(BaseModel):
    message: str

#creates a URL endpoint
@app.post("/chat")

def chat(req: ChatRequest):
    global chat_history

    history = [{"role": h["role"], "content": h["content"]} for h in chat_history]
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": req.message}]
    )
    response = client.chat.completions.create(model=MODEL, messages=messages, tools = tools)
    reply = response.choices[0].message.content
    if hasattr(reply, "tool_calls") and reply.tool_calls:

        tool_call = reply.tool_calls[0]
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        if function_name == "get_ticket_price":
            result = get_ticket_price(arguments["destination_city"])

            messages.append(reply)
            messages.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id
            })

            second_response = client.chat.completions.create(
                model=MODEL,
                messages=messages
            )

            reply = second_response.choices[0].message.content

    else:
        reply = response_message.content

    # Save chat
    chat_history.append({"role": "user", "content": req.message})
    chat_history.append({"role": "assistant", "content": reply})

    return {"reply": reply}

    




In [4]:
import requests
from dotenv import load_dotenv
import os
import json

load_dotenv(override=True)

url = "https://booking-com18.p.rapidapi.com/flights/v2/search-roundtrip"

querystring = {
    "departId": "JFK",
    "arrivalId": "LOS",
    "departDate": "2026-04-15",   # ← add this
    "returnDate": "2026-04-22",   # ← add this
    "adults": "1",
    "currency_code": "USD"
}

headers = {
    "x-rapidapi-key": os.getenv("XRAPID_API_KEY"),
    "x-rapidapi-host": "booking-com18.p.rapidapi.com",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers, params=querystring)
print(json.dumps(response.json(), indent=2))

{
  "data": {
    "aggregation": {
      "totalCount": 1002,
      "filteredTotalCount": 1002,
      "stops": [
        {
          "numberOfStops": 1,
          "count": 342,
          "minPrice": {
            "currencyCode": "USD",
            "units": 1209,
            "nanos": 680000000
          },
          "minPriceRound": {
            "currencyCode": "USD",
            "units": 1210,
            "nanos": 0
          },
          "minPricePerAdult": {
            "currencyCode": "USD",
            "units": 1210,
            "nanos": 0
          },
          "cheapestAirline": {
            "name": "Royal Air Maroc",
            "code": "AT",
            "logo": "https://r-xx.bstatic.com/data/airlines_logo/AT.png"
          }
        },
        {
          "numberOfStops": 2,
          "count": 794,
          "minPrice": {
            "currencyCode": "USD",
            "units": 1209,
            "nanos": 680000000
          },
          "minPriceRound": {
            "currencyC